# Streamlined Analysis Template

This notebook demonstrates how to use the new `ExperimentResults` and `AttributionAnalyzer` for scalable, high-performance EEG attribution analysis.

In [ ]:
import os
import pickle
import sys


import pandas as pd

# Add src to path
sys.path.append(os.path.abspath("../src"))

from evaluate.analysis_utils import ExperimentResults, AttributionAnalyzer
from visualization.eeg_attribution import attribution_topomap, plot_attribution_summary
from evaluate.analyze import plot_summary_grid

import logging
import sys
import seaborn as sns
sns.set_theme('paper', style='white', rc={
    'font.size': 10,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'figure.figsize': (5.5, 3) # textwidth
})
import re

# 1. Setup the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 2. Configure basicConfig to use stdout
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

logger.info("This log should now appear in the notebook cell!")
import mne
print(mne.__version__)

## 1. Load Experiment Results
Point to your experiment directory (the one containing `run_*` folders and `evaluation_summary.csv`).

## 2. Calculate Global Statistics (Full Dataset)
Unlike previous notebooks, this uses **Dask** to process the entire dataset without sampling.

In [ ]:
EXPERIMENT_DIR_MAP = {
    'frozen': '$HOME/Downloads/physionet_binary_frozen_evaluation',
    'from scratch': '$HOME/Downloads/true_binary_physionet_from_scratch_evaluation',
    'finetuned': '$HOME/Downloads/true_binary_physionet_pretrained_evaluation',

}

experiment_statistics_map = {}

for experiment, dir in EXPERIMENT_DIR_MAP.items():
    results = ExperimentResults(dir)
    performance = results.performance_df
    analyzer = AttributionAnalyzer(results)
    print(f"Discovered {len(results.run_dirs)} runs.")
    run_statistics = {}
    run_ids_ordered = [int(re.match(r'.*run_(\d+).*', run_dir).group(1)) for run_dir in results.run_dirs]
    print(run_ids_ordered)
    print(results.run_dirs)
    for run_id in run_ids_ordered: # Ensures statistic corresponds to 'actual' run id
        ddf = analyzer.load_run_data(run_id, split='test')
        statistics = AttributionAnalyzer.calculate_statistics(ddf=ddf)
        run_statistics[run_id] = statistics
    experiment_statistics_map[experiment] = {'performance': performance, 'analyzer': analyzer, 'results': results, 'statistics': run_statistics}
experiment_statistics_map

In [ ]:
experiment_statistics_map['finetuned']['statistics'][0]

In [ ]:
with open("../../out/physiomi_lr_attribution_statistics.pkl", "wb") as f:
    pickle.dump(
        experiment_statistics_map,
        f,
    )


## 3. Visualization
Use the aggregated statistics for plotting.

In [ ]:
# For further analysis, you may want to load the dataset corresponding to your run
from data.physionet_datamodule import PhysionetDatamodule
from utils import adapt_channel_names_to_mne
datamodule = PhysionetDatamodule(dataset_path="data/physiomi", data_dir="$HOME/Projects/datasets/physiomi/", classes=['left_hand', 'right_hand'])
# datamodule.prepare_data()
datamodule.setup()
dataset = datamodule.test_ds
print(dataset)
ch_names = adapt_channel_names_to_mne(dataset[0]['eeg_channels'])
print(ch_names)

In [ ]:
experiment_statistics_map['from scratch']['statistics'][3].keys()

In [ ]:
# Example: Plot mean attribution for class 0
experiments = list(experiment_statistics_map.keys())
run_ids = list(experiment_statistics_map[experiments[0]]['statistics'].keys()) # Get run id's from an example statistic
# build the attribution grid
titles = [[f"B.acc.{experiment_statistics_map[exp]['performance'].at[run, 'test_balanced_accuracy']*100:.2f}%" for run in run_ids] for exp in experiments]
# Ensure consistent iteration order across experiment runs.
# Taking class 1 = left hand doesn't always work due to some collapsed models.
attributions = []
for exp in experiments:
    exp_attributions = []
    for run in run_ids:
        exp_stats = experiment_statistics_map[exp]['statistics'][run]
        if len(exp_stats) == 1:
            attr = exp_stats[0]['mean']
        elif len(exp_stats) == 2:
            # merge means
            attr = (exp_stats[0]['mean'] * exp_stats[0]['count'] + exp_stats[1]['mean'] * exp_stats[1]['count']) / (exp_stats[0]['count'] + exp_stats[1]['count'])
        else:
            raise ValueError(f"Unexpected number of statistics: {len(exp_stats)}")
        exp_attributions.append(attr)
    attributions.append(exp_attributions)

print(titles)
#attributions = [experiment_statistics_map[exp]['statistics'][1] for exp in experiment_statistics_map.keys()]
fig = plot_summary_grid(
    attributions=attributions,
    titles=titles,
    channel_names=ch_names,
    suptitle=None,
    supylabel=f'bottom: {experiments[-1]} | middle: {experiments[-2]} | top: {experiments[-3]}',
    supxlabel=f'runs',
    figsize=(5.5, 5),
    spatial_to_time_ratio=9
)

# Strip x-axis ticks and labels from all subplots except the bottom row
# for ax in fig.axes[:-len(run_ids)]:  # Exclude last n axes which correspond to bottom row
#    ax.tick_params(axis='x', labelbottom=False)

fig.savefig('../../out/all_physionetlr.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# Creating pretty performance tables
perf_dfs = {k: experiment_statistics_map[k]['performance'] for k in experiment_statistics_map.keys()}
for k, df in perf_dfs.items():
    df['model config'] = str(k)
all_performances = pd.concat(perf_dfs.values(), ignore_index=True)
not_interesting = ["test_loss", "seed", "run"]
rename_map = {metric: metric.replace("test_", "") for metric in all_performances.columns if metric not in not_interesting}

grouped = all_performances.drop(columns=not_interesting).rename(columns=rename_map).groupby(["model config"]).agg(
["mean", "std"])
formatted = pd.DataFrame(index=grouped.index)
for metric in list(grouped.columns.levels[0]):
    if not "accuracy" in metric:
        formatted[metric] = grouped.apply(lambda grouped: f'{grouped[(metric, "mean")]:.4f} ± {grouped[(metric, "std")]:.4f}', axis=1)
    else:
        formatted[metric] = grouped.apply(lambda grouped: f'{grouped[(metric, "mean")]*100:.2f}% ± {grouped[(metric, "std")]*100:.2f}', axis=1)
formatted.to_clipboard(excel=True)
formatted
